# IBD Post-NMF Analysis

Parameterized Post-NMF notebook for NicheRunner.

In [ ]:
from pathlib import Path
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.sparse import csr_matrix, issparse
from scipy.stats import mannwhitneyu
from sklearn.feature_selection import mutual_info_classif
from sklearn.neighbors import BallTree
from statsmodels.stats.multitest import multipletests


In [ ]:
# Papermill parameters
REFERENCE_H5AD = ""
COSMX_H5AD = ""
COSMX_WITH_NMF_H5AD = ""
CELL_METADATA_CSV = ""
OUTPUT_DIR = ""
RUN_DIR = ""


In [ ]:
reference_h5ad_path = REFERENCE_H5AD
cosmx_h5ad_path = COSMX_H5AD
cosmx_with_nmf_h5ad_path = COSMX_WITH_NMF_H5AD
cell_metadata_path = CELL_METADATA_CSV
output_dir = Path(OUTPUT_DIR or "post_nmf_outputs").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

reference_path = Path(reference_h5ad_path) if reference_h5ad_path else None
cosmx_path = Path(cosmx_with_nmf_h5ad_path or cosmx_h5ad_path)
metadata_path = Path(cell_metadata_path) if cell_metadata_path else None

if not cosmx_path.exists():
    raise FileNotFoundError(f"CosMx AnnData not found: {cosmx_path}")

def _build_unique_cell_id(frame, cell_column):
    if "unique_cell_id" in frame.columns:
        return frame["unique_cell_id"].astype(str)
    if "fov" in frame.columns and cell_column in frame.columns:
        return frame["fov"].astype(str) + "__" + frame[cell_column].astype(str)
    return frame.index.astype(str)

def _read_metadata(path):
    compression = "gzip" if path.suffix == ".gz" else "infer"
    return pd.read_csv(path, compression=compression)

def _merge_metadata(obs, metadata):
    merged = obs.copy()
    if metadata is None or metadata.empty:
        return merged

    metadata = metadata.copy()
    meta_cell_col = "cell_ID" if "cell_ID" in metadata.columns else "cell_id" if "cell_id" in metadata.columns else None
    obs_cell_col = "cell_id" if "cell_id" in merged.columns else "cell_ID" if "cell_ID" in merged.columns else None

    if "unique_cell_id" in merged.columns and "unique_cell_id" in metadata.columns:
        left_key = merged["unique_cell_id"].astype(str)
        right_key = metadata["unique_cell_id"].astype(str)
    elif "fov" in merged.columns and obs_cell_col and "fov" in metadata.columns and meta_cell_col:
        left_key = merged["fov"].astype(str) + "__" + merged[obs_cell_col].astype(str)
        right_key = metadata["fov"].astype(str) + "__" + metadata[meta_cell_col].astype(str)
    else:
        return merged

    metadata = metadata.assign(_join_key=right_key)
    metadata = metadata.drop_duplicates("_join_key")
    merged = merged.assign(_join_key=left_key)
    metadata_only = metadata.drop(columns=[col for col in metadata.columns if col in merged.columns and col != "_join_key"])
    merged = merged.merge(metadata_only, on="_join_key", how="left")
    return merged.drop(columns=["_join_key"])

def _compute_area(frame):
    if "Area" in frame.columns:
        area = pd.to_numeric(frame["Area"], errors="coerce")
        if area.notna().any():
            return area
    width_col = "Width" if "Width" in frame.columns else "width" if "width" in frame.columns else None
    height_col = "Height" if "Height" in frame.columns else "height" if "height" in frame.columns else None
    if width_col and height_col:
        width = pd.to_numeric(frame[width_col], errors="coerce")
        height = pd.to_numeric(frame[height_col], errors="coerce")
        area = width * height
        if area.notna().any():
            return area
    return pd.Series(np.full(len(frame), np.pi * (15.0 / 2.0) ** 2), index=frame.index, dtype=float)

def _pick_label(frame, candidates, default):
    for name in candidates:
        if name in frame.columns:
            values = frame[name].astype(object)
            values = values.where(pd.notna(values), np.nan)
            values = values.replace("nan", np.nan)
            if values.notna().any():
                return values if default is None else values.fillna(default)
    return pd.Series(np.nan if default is None else default, index=frame.index, dtype=object)

def _first_non_null(values, fallback):
    values = pd.Series(values)
    values = values.dropna()
    return values.iloc[0] if len(values) else fallback

def _sorted_factor_labels(values):
    labels = []
    for value in values:
        text = str(value)
        if text.lower() in {"nan", "none", "unassigned"}:
            continue
        labels.append(text)
    def _sort_key(label):
        return (0, int(label)) if label.isdigit() else (1, label)
    return sorted(set(labels), key=_sort_key)

def _choose_expression_matrix(adata_obj):
    if adata_obj.raw is not None:
        return adata_obj.raw.X, pd.Index(adata_obj.raw.var_names.astype(str))
    return adata_obj.X, pd.Index(adata_obj.var_names.astype(str))

def _group_mean_expression(matrix, group_codes, n_groups):
    rows = np.arange(len(group_codes))
    selector = csr_matrix((np.ones(len(group_codes), dtype=float), (group_codes, rows)), shape=(n_groups, len(group_codes)))
    if issparse(matrix):
        grouped = selector @ matrix.tocsr()
        grouped = grouped.toarray()
    else:
        grouped = selector @ np.asarray(matrix)
    counts = np.bincount(group_codes, minlength=n_groups).astype(float)
    counts[counts == 0] = 1.0
    return grouped / counts[:, None]


In [ ]:
adata = sc.read_h5ad(cosmx_path)
obs = adata.obs.copy()
obs["unique_cell_id"] = _build_unique_cell_id(obs, "cell_id" if "cell_id" in obs.columns else "cell_ID" if "cell_ID" in obs.columns else adata.obs.index.name or "index")

metadata_df = _read_metadata(metadata_path) if metadata_path and metadata_path.exists() else None
obs = _merge_metadata(obs, metadata_df)
obs["Area"] = _compute_area(obs)
obs["diameter_um"] = 2.0 * np.sqrt(obs["Area"] / np.pi)
obs["nmf_factor"] = _pick_label(obs, ["NMF_factor", "dominant_nmf_factor", "cell_type"], "unassigned")
obs["patient"] = _pick_label(obs, ["patient", "Patient", "sample", "Sample"], "unknown_patient")
obs["disease_state"] = _pick_label(obs, ["Disease_State", "disease_state"], None)
missing_disease = obs["disease_state"].isna()
obs.loc[missing_disease, "disease_state"] = obs.loc[missing_disease, "patient"].astype(str).str[:2]
obs["fov_key"] = _pick_label(obs, ["unique_fov"], None)
missing_fov = obs["fov_key"].isna()
if "fov" in obs.columns:
    obs.loc[missing_fov, "fov_key"] = obs.loc[missing_fov, "patient"].astype(str) + "_" + obs.loc[missing_fov, "fov"].astype(str)
obs["fov_key"] = obs["fov_key"].fillna(obs["patient"].astype(str))

ref_summary = {}
if reference_path and reference_path.exists():
    reference_adata = sc.read_h5ad(reference_path, backed="r")
    ref_summary = {
        "reference_path": str(reference_path),
        "reference_n_obs": int(reference_adata.n_obs),
        "reference_n_vars": int(reference_adata.n_vars),
        "shared_genes": int(len(set(reference_adata.var_names).intersection(set(adata.var_names)))),
    }
    if getattr(reference_adata, "file", None) is not None:
        reference_adata.file.close()

obs.to_csv(output_dir / "post_nmf_obs.csv", index=True)
obs.head(500).to_csv(output_dir / "post_nmf_obs_preview.csv", index=True)

summary = {
    "cosmx_path": str(cosmx_path),
    "metadata_path": str(metadata_path) if metadata_path and metadata_path.exists() else None,
    "n_cells": int(adata.n_obs),
    "n_genes": int(adata.n_vars),
    "n_patients": int(obs["patient"].nunique()),
    "n_fovs": int(obs["fov_key"].nunique()),
    "nmf_factor_count": int(obs["nmf_factor"].nunique()),
    "area_missing_fraction": float(obs["Area"].isna().mean()),
}
summary.update(ref_summary)
(output_dir / "post_nmf_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

plt.figure(figsize=(8, 5))
sns.histplot(obs["diameter_um"].dropna(), bins=40)
plt.xlabel("Estimated cell diameter")
plt.ylabel("Cells")
plt.tight_layout()
plt.savefig(output_dir / "cell_diameter_histogram.png", dpi=200, bbox_inches="tight")
plt.close()


In [ ]:
numeric_candidates = [
    "diameter_um",
    "Area",
    "CenterX_global_px",
    "CenterY_global_px",
    "center_x",
    "center_y",
]
numeric_cols = [col for col in numeric_candidates if col in obs.columns]

aggregations = {}
for col in numeric_cols:
    series = pd.to_numeric(obs[col], errors="coerce")
    obs[col] = series
    aggregations[f"{col}_mean"] = (col, "mean")
    aggregations[f"{col}_std"] = (col, "std")

# Keep morphology summaries available for inspection, but do not propagate them into the
# final classifier feature pool for this non-cancer setting.
grouped = obs.groupby("patient").agg(**aggregations) if aggregations else pd.DataFrame(index=sorted(obs["patient"].unique()))
grouped["cell_count"] = obs.groupby("patient").size()
grouped["fov_count"] = obs.groupby("patient")["fov_key"].nunique()

def _compute_fov_neighborhood_enrichment(obs_df):
    required = {"CenterX_global_px", "CenterY_global_px", "Area", "nmf_factor", "fov_key", "disease_state", "patient"}
    missing = required - set(obs_df.columns)
    if missing:
        raise RuntimeError(f"Missing neighborhood-enrichment columns: {sorted(missing)}")
    coords = obs_df[["CenterX_global_px", "CenterY_global_px"]].astype(float).to_numpy()
    diameters = 2 * np.sqrt(pd.to_numeric(obs_df["Area"], errors="coerce") / np.pi)
    labels = obs_df["nmf_factor"].astype(str)
    factors = sorted(labels.unique(), key=lambda v: (len(v), v))
    rows = []
    for fov_id, index in obs_df.groupby("fov_key").groups.items():
        idx = np.asarray(index)
        fov_coords = coords[idx]
        if len(fov_coords) < 2:
            continue
        fov_diams = diameters.iloc[idx].to_numpy()
        fov_labels = labels.iloc[idx].reset_index(drop=True)
        tree = BallTree(fov_coords)
        interaction_matrix = pd.DataFrame(0.0, index=factors, columns=factors)
        for i in range(len(fov_coords)):
            radius = 2 * fov_diams[i]
            neighbor_idx = tree.query_radius([fov_coords[i]], r=radius)[0]
            neighbor_idx = neighbor_idx[neighbor_idx != i]
            if len(neighbor_idx) == 0:
                continue
            factor_i = fov_labels.iloc[i]
            factor_j_counts = fov_labels.iloc[neighbor_idx].value_counts()
            for factor_j, count in factor_j_counts.items():
                interaction_matrix.loc[factor_i, factor_j] += float(count)
        interaction_matrix = interaction_matrix + interaction_matrix.T
        niche_props = fov_labels.value_counts(normalize=True).reindex(factors, fill_value=0.0)
        total_interactions = float(interaction_matrix.to_numpy().sum())
        expected = total_interactions * np.outer(niche_props, niche_props)
        expected_df = pd.DataFrame(expected, index=factors, columns=factors)
        enrichment = np.log2((interaction_matrix + 1.0) / (expected_df + 1.0))
        row = {
            "field_of_view": str(fov_id),
            "patient": str(obs_df.loc[idx[0], "patient"]),
            "Disease_State": str(obs_df.loc[idx[0], "disease_state"]),
        }
        for ii, fi in enumerate(factors, 1):
            for jj, fj in enumerate(factors, 1):
                row[f"enrichment_{ii}-{jj}"] = float(enrichment.loc[fi, fj])
        rows.append(row)
    return pd.DataFrame(rows)

nmf_counts = pd.crosstab(obs["patient"], obs["nmf_factor"]).sort_index()
nmf_props = nmf_counts.div(nmf_counts.sum(axis=1), axis=0).fillna(0.0)
nmf_props.columns = [f"nmf_prop_{col}" for col in nmf_props.columns]

fov_disease = obs.groupby("fov_key")["disease_state"].agg(lambda values: _first_non_null(values, "unknown"))
fov_patient = obs.groupby("fov_key")["patient"].agg(lambda values: _first_non_null(values, "unknown_patient"))
fov_index = pd.Index(sorted(fov_disease.index.astype(str)), name="fov_key")
fov_nmf_counts = pd.crosstab(obs["fov_key"], obs["nmf_factor"]).reindex(fov_index, fill_value=0)
fov_nmf_props = fov_nmf_counts.div(fov_nmf_counts.sum(axis=1), axis=0).fillna(0.0)
fov_nmf_props.columns = [f"nmf_prop_{col}" for col in fov_nmf_props.columns]

# Spatial neighborhood-enrichment features for classifier input.
enrichment_fov = _compute_fov_neighborhood_enrichment(obs)
if enrichment_fov.empty:
    raise RuntimeError("Neighborhood-enrichment feature construction produced no FOV-level rows.")
enrichment_fov = enrichment_fov.set_index("field_of_view").sort_index()
enrichment_feature_cols = [c for c in enrichment_fov.columns if c.startswith("enrichment_")]
enrichment_fov[enrichment_feature_cols].to_parquet(output_dir / "enrichment_features_fov.parquet")
enrichment_fov[enrichment_feature_cols].to_csv(output_dir / "enrichment_features_fov.csv")

enrichment_metadata = enrichment_fov[["patient", "Disease_State"]].copy()
mi_enrichment_table = pd.DataFrame(columns=["feature", "mi_score"])
significant_enrichment = pd.DataFrame(columns=["feature", "mi_score", "group_1", "group_2", "group_1_mean", "group_2_mean", "direction", "p_raw", "p_adj"])
selected_enrichment_features = []
if enrichment_feature_cols and enrichment_metadata["Disease_State"].nunique() > 1:
    disease_codes = pd.Categorical(enrichment_metadata["Disease_State"])
    enrichment_matrix = enrichment_fov[enrichment_feature_cols].fillna(0.0)
    mi_scores = mutual_info_classif(enrichment_matrix.to_numpy(), disease_codes.codes, random_state=42)
    mi_enrichment_table = pd.DataFrame({"feature": enrichment_feature_cols, "mi_score": mi_scores})
    mi_enrichment_table = mi_enrichment_table.sort_values(["mi_score", "feature"], ascending=[False, True]).reset_index(drop=True)
    mi_enrichment_table.to_csv(output_dir / "enrichment_mi_scores.csv", index=False)

    candidate_count = min(60, len(mi_enrichment_table))
    candidates = mi_enrichment_table.head(candidate_count).copy()
    tests = []
    groups_present = list(disease_codes.categories)
    group_1, group_2 = groups_present[0], groups_present[1]
    for feature, mi_score in candidates[["feature", "mi_score"]].itertuples(index=False):
        x = enrichment_matrix.loc[enrichment_metadata["Disease_State"] == group_1, feature]
        y = enrichment_matrix.loc[enrichment_metadata["Disease_State"] == group_2, feature]
        if x.nunique() == 0 and y.nunique() == 0:
            continue
        _, p_raw = mannwhitneyu(x, y, alternative="two-sided")
        group_1_mean = float(x.mean())
        group_2_mean = float(y.mean())
        tests.append({
            "feature": feature,
            "mi_score": float(mi_score),
            "group_1": group_1,
            "group_2": group_2,
            "group_1_mean": group_1_mean,
            "group_2_mean": group_2_mean,
            "direction": "Up" if group_1_mean > group_2_mean else "Down",
            "p_raw": float(p_raw),
        })
    if tests:
        significant_enrichment = pd.DataFrame(tests)
        significant_enrichment["p_adj"] = multipletests(significant_enrichment["p_raw"], method="fdr_bh")[1]
        significant_enrichment = significant_enrichment.sort_values(["p_adj", "mi_score"], ascending=[True, False]).reset_index(drop=True)
        significant_enrichment.to_csv(output_dir / "enrichment_significant_features.csv", index=False)
        selected_enrichment_features = significant_enrichment.loc[significant_enrichment["p_adj"] < 0.05, "feature"].tolist()
        if not selected_enrichment_features:
            selected_enrichment_features = significant_enrichment["feature"].head(min(10, len(significant_enrichment))).tolist()
    else:
        mi_enrichment_table.to_csv(output_dir / "enrichment_mi_scores.csv", index=False)

patient_enrichment = pd.DataFrame(index=sorted(obs["patient"].astype(str).unique()))
if selected_enrichment_features:
    patient_enrichment = enrichment_fov[selected_enrichment_features].join(enrichment_metadata[["patient"]]).groupby("patient").mean()
    patient_enrichment.index.name = "patient"
    patient_enrichment.to_parquet(output_dir / "enrichment_features_patient.parquet")
    patient_enrichment.to_csv(output_dir / "enrichment_features_patient.csv")

expr_matrix, gene_names = _choose_expression_matrix(adata)
factor_labels = _sorted_factor_labels(obs["nmf_factor"])
niche_gene_frames = []
for factor in factor_labels:
    factor_mask = obs["nmf_factor"].astype(str) == factor
    if factor_mask.sum() == 0:
        continue
    factor_fovs = pd.Categorical(obs.loc[factor_mask, "fov_key"].astype(str), categories=fov_index)
    valid = factor_fovs.codes >= 0
    if not np.any(valid):
        continue
    factor_means = _group_mean_expression(expr_matrix[factor_mask.to_numpy()][valid], factor_fovs.codes[valid], len(fov_index))
    factor_df = pd.DataFrame(
        factor_means,
        index=fov_index,
        columns=[f"niche_{factor}_gene_{gene}" for gene in gene_names],
    )
    niche_gene_frames.append(factor_df)

if niche_gene_frames:
    niche_gene_fov = pd.concat(niche_gene_frames, axis=1)
    niche_gene_fov = niche_gene_fov.loc[:, niche_gene_fov.var(axis=0) > 0]
else:
    niche_gene_fov = pd.DataFrame(index=fov_index)

niche_gene_metadata = pd.DataFrame({
    "field_of_view": fov_index,
    "patient": fov_patient.reindex(fov_index).astype(str).values,
    "Disease_State": fov_disease.reindex(fov_index).astype(str).values,
}).set_index("field_of_view")
niche_gene_fov.to_parquet(output_dir / "niche_gene_features_fov.parquet")
niche_gene_fov.to_csv(output_dir / "niche_gene_features_fov.csv")

mi_table = pd.DataFrame(columns=["feature", "mi_score", "niche", "gene"])
selected_niche_gene_features = []
significant_niche_gene = pd.DataFrame(columns=["feature", "mi_score", "niche", "gene", "group_1", "group_2", "group_1_mean", "group_2_mean", "direction", "p_raw", "p_adj"])
if not niche_gene_fov.empty and niche_gene_metadata["Disease_State"].nunique() > 1:
    disease_codes = pd.Categorical(niche_gene_metadata["Disease_State"])
    mi_scores = mutual_info_classif(niche_gene_fov.to_numpy(), disease_codes.codes, random_state=42)
    mi_table = pd.DataFrame({"feature": niche_gene_fov.columns, "mi_score": mi_scores})
    split_feature = mi_table["feature"].str.extract(r"^niche_(.+?)_gene_(.+)$")
    mi_table["niche"] = split_feature[0]
    mi_table["gene"] = split_feature[1]
    mi_table = mi_table.sort_values(["mi_score", "feature"], ascending=[False, True]).reset_index(drop=True)
    mi_table.to_csv(output_dir / "niche_gene_mi_scores.csv", index=False)

    candidate_count = min(60, len(mi_table))
    candidates = mi_table.head(candidate_count).copy()
    tests = []
    groups_present = list(disease_codes.categories)
    group_1, group_2 = groups_present[0], groups_present[1]
    for feature, mi_score, niche, gene in candidates[["feature", "mi_score", "niche", "gene"]].itertuples(index=False):
        x = niche_gene_fov.loc[niche_gene_metadata["Disease_State"] == group_1, feature]
        y = niche_gene_fov.loc[niche_gene_metadata["Disease_State"] == group_2, feature]
        if x.nunique() == 0 and y.nunique() == 0:
            continue
        _, p_raw = mannwhitneyu(x, y, alternative="two-sided")
        group_1_mean = float(x.mean())
        group_2_mean = float(y.mean())
        tests.append({
            "feature": feature,
            "mi_score": float(mi_score),
            "niche": niche,
            "gene": gene,
            "group_1": group_1,
            "group_2": group_2,
            "group_1_mean": group_1_mean,
            "group_2_mean": group_2_mean,
            "direction": "Up" if group_1_mean > group_2_mean else "Down",
            "p_raw": float(p_raw),
        })
    if tests:
        significant_niche_gene = pd.DataFrame(tests)
        significant_niche_gene["p_adj"] = multipletests(significant_niche_gene["p_raw"], method="fdr_bh")[1]
        significant_niche_gene = significant_niche_gene.sort_values(["p_adj", "mi_score"], ascending=[True, False]).reset_index(drop=True)
        significant_niche_gene.to_csv(output_dir / "niche_gene_significant_features.csv", index=False)
        selected_niche_gene_features = significant_niche_gene.loc[significant_niche_gene["p_adj"] < 0.05, "feature"].tolist()
        if not selected_niche_gene_features:
            selected_niche_gene_features = significant_niche_gene["feature"].head(min(10, len(significant_niche_gene))).tolist()
    else:
        mi_table.to_csv(output_dir / "niche_gene_mi_scores.csv", index=False)

patient_niche_gene = pd.DataFrame(index=sorted(obs["patient"].astype(str).unique()))
if selected_niche_gene_features:
    patient_niche_gene = niche_gene_fov[selected_niche_gene_features].join(niche_gene_metadata[["patient"]]).groupby("patient").mean()
    patient_niche_gene.index.name = "patient"
    patient_niche_gene.to_parquet(output_dir / "niche_gene_features_patient.parquet")
    patient_niche_gene.to_csv(output_dir / "niche_gene_features_patient.csv")

disease_targets = obs.groupby("patient")["disease_state"].agg(lambda values: values.dropna().iloc[0] if values.dropna().shape[0] else "unknown")
groups = pd.Series(disease_targets.index, index=disease_targets.index, name="patient")

# Morphology summaries are exported separately but not included in the classifier feature pool.
combined = nmf_props.join(patient_enrichment, how="left").join(patient_niche_gene, how="left").fillna(0.0)
combined.index.name = "patient"
combined.to_parquet(output_dir / "combined_features_filtered.parquet")
combined.to_csv(output_dir / "combined_features_filtered.csv")
grouped.to_parquet(output_dir / "morphology_summary_features.parquet")
grouped.to_csv(output_dir / "morphology_summary_features.csv")

feature_columns = list(combined.columns)
if not feature_columns:
    raise RuntimeError("No patient-level features were generated from the Post-NMF inputs.")
priority_columns = [col for col in nmf_props.columns if col in combined.columns]
remaining_slots = max(0, min(15, len(feature_columns)) - len(priority_columns))
if remaining_slots > 0:
    selected_enrichment_in_combined = [col for col in selected_enrichment_features if col in combined.columns]
    selected_niche_gene_in_combined = [col for col in selected_niche_gene_features if col in combined.columns]
    enrichment_quota = min(len(selected_enrichment_in_combined), remaining_slots // 2 if selected_niche_gene_in_combined else remaining_slots)
    niche_quota = min(len(selected_niche_gene_in_combined), remaining_slots - enrichment_quota)
    priority_columns.extend([col for col in selected_enrichment_in_combined if col not in priority_columns][:enrichment_quota])
    priority_columns.extend([col for col in selected_niche_gene_in_combined if col not in priority_columns][:niche_quota])
    if len(priority_columns) < min(15, len(feature_columns)):
        remaining = [col for col in feature_columns if col not in priority_columns]
        remaining = sorted(remaining, key=lambda col: float(combined[col].var()), reverse=True)
        priority_columns.extend(remaining)
selected_columns = priority_columns[:min(15, len(priority_columns))]
reduced = combined[selected_columns].copy()
reduced.to_parquet(output_dir / "reduced_features_final_15.parquet")
reduced.to_csv(output_dir / "reduced_features_final_15.csv")

disease_targets.to_frame("Disease_State").to_parquet(output_dir / "targets_y.parquet")
groups.to_frame("patient").to_parquet(output_dir / "groups.parquet")
patient_nmf = nmf_props.copy()
if not patient_nmf.empty:
    plt.figure(figsize=(max(6, patient_nmf.shape[1] * 0.6), max(4, patient_nmf.shape[0] * 0.4)))
    sns.heatmap(patient_nmf, cmap="viridis")
    plt.xlabel("NMF factor")
    plt.ylabel("Patient")
    plt.tight_layout()
    plt.savefig(output_dir / "patient_nmf_proportions.png", dpi=200, bbox_inches="tight")
    plt.close()

artifacts = {
    "reduced_features": str(output_dir / "reduced_features_final_15.parquet"),
    "targets": str(output_dir / "targets_y.parquet"),
    "groups": str(output_dir / "groups.parquet"),
    "combined_features": str(output_dir / "combined_features_filtered.parquet"),
    "morphology_summary": str(output_dir / "morphology_summary_features.parquet"),
    "enrichment_fov": str(output_dir / "enrichment_features_fov.parquet"),
    "enrichment_patient": str(output_dir / "enrichment_features_patient.parquet") if not patient_enrichment.empty else None,
    "enrichment_mi_scores": str(output_dir / "enrichment_mi_scores.csv"),
    "enrichment_significant": str(output_dir / "enrichment_significant_features.csv"),
    "niche_gene_fov": str(output_dir / "niche_gene_features_fov.parquet"),
    "niche_gene_patient": str(output_dir / "niche_gene_features_patient.parquet") if not patient_niche_gene.empty else None,
    "niche_gene_mi_scores": str(output_dir / "niche_gene_mi_scores.csv"),
    "niche_gene_significant": str(output_dir / "niche_gene_significant_features.csv"),
}
(output_dir / "post_nmf_artifacts.json").write_text(json.dumps(artifacts, indent=2), encoding="utf-8")
artifacts
